In [1]:
# During the review process of the paper submitted to ICML 2025, 
# a reviewer (referred to as Reviewer t5C9) introduced an alternative algorithm/code 
# for computing the Min-Max-Jump (MMJ) distance matrix. See:
# https://openreview.net/forum?id=qNfEkSuGKk

# This file tests a C++ implementation of Reviewer t5C9's algorithm. 
# The Python-to-C++ conversion was performed using ChatGPT.

In [2]:
%%writefile Reviewer_t5C9_code_cpp_version.cpp

#include <iostream>
#include <vector>
#include <queue>
#include <limits>
#include <random>
#include <chrono>
#include <algorithm>
#include <list>
#include <iomanip>



using namespace std;

struct Edge {
    int u, v;
    double weight;
};

// Prim's MST (dense graph version)
vector<Edge> prim_mst(const vector<vector<double>>& D) {
    int n = D.size();
    vector<bool> in_mst(n, false);
    vector<double> key(n, numeric_limits<double>::infinity());
    vector<int> parent(n, -1);
    key[0] = 0.0;

    for (int count = 0; count < n; ++count) {
        double min_val = numeric_limits<double>::infinity();
        int u = -1;
        for (int i = 0; i < n; ++i)
            if (!in_mst[i] && key[i] < min_val)
                min_val = key[i], u = i;

        if (u == -1) break;

        in_mst[u] = true;
        for (int v = 0; v < n; ++v)
            if (!in_mst[v] && D[u][v] < key[v])
                key[v] = D[u][v], parent[v] = u;
    }

    vector<Edge> edges;
    for (int v = 1; v < n; ++v)
        edges.push_back({parent[v], v, D[parent[v]][v]});
    return edges;
}

// Build CSR-like structure
void build_csr(int n, const vector<Edge>& mst_edges,
               vector<int>& ptr, vector<int>& adj_edges, vector<double>& adj_weights) {
    vector<int> edge_counts(n, 0);
    for (const auto& e : mst_edges) {
        edge_counts[e.u]++;
        edge_counts[e.v]++;
    }

    ptr.resize(n + 1);
    for (int i = 1; i <= n; ++i)
        ptr[i] = ptr[i - 1] + edge_counts[i - 1];

    adj_edges.resize(ptr[n]);
    adj_weights.resize(ptr[n]);
    vector<int> positions(n, 0);

    for (const auto& e : mst_edges) {
        for (int i = 0; i < 2; ++i) {
            int u = (i == 0) ? e.u : e.v;
            int v = (i == 0) ? e.v : e.u;
            int idx = ptr[u] + positions[u]++;
            adj_edges[idx] = v;
            adj_weights[idx] = e.weight;
        }
    }
}


#include <tbb/parallel_for.h>
#include <tbb/blocked_range.h>
#include <tbb/parallel_for_each.h>

vector<vector<double>> compute_bottleneck_matrix(int n,
    const vector<int>& ptr, const vector<int>& adj_edges, const vector<double>& adj_weights) {

    vector<vector<double>> bottleneck(n, vector<double>(n, 0.0));

    tbb::parallel_for(tbb::blocked_range<int>(0, n),
        [&](const tbb::blocked_range<int>& r) {
            for (int src = r.begin(); src < r.end(); ++src) {
                vector<bool> visited(n, false);
                vector<double> max_edges(n, 0.0);
                queue<pair<int, double>> q;

                visited[src] = true;
                q.push({src, 0.0});

                while (!q.empty()) {
                    auto [u, curr_max] = q.front(); q.pop();

                    for (int i = ptr[u]; i < ptr[u + 1]; ++i) {
                        int v = adj_edges[i];
                        double weight = adj_weights[i];

                        if (!visited[v]) {
                            double new_max = max(curr_max, weight);
                            visited[v] = true;
                            max_edges[v] = new_max;
                            q.push({v, new_max});
                        }
                    }
                }

                bottleneck[src] = move(max_edges); // safe: each src owns its row
            }
        }
    );

    return bottleneck;
}


vector<vector<double>> ultra_fast_wide(const vector<vector<double>>& D) {
    int n = D.size();


    vector<Edge> mst = prim_mst(D);

    vector<int> ptr, adj_edges;
    vector<double> adj_weights;
    build_csr(n, mst, ptr, adj_edges, adj_weights);

    return compute_bottleneck_matrix(n, ptr, adj_edges, adj_weights);
}

vector<vector<double>> create_symmetric_distance_matrix(int N, int seed) {
    mt19937 gen(seed);
    uniform_int_distribution<> dist(1, 9999);
    vector<vector<double>> A(N, vector<double>(N));
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
            A[i][j] = dist(gen);

    vector<vector<double>> sym_A(N, vector<double>(N));
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j)
            sym_A[i][j] = floor((A[i][j] + A[j][i]) / 2.0);

    for (int i = 0; i < N; ++i)
        sym_A[i][i] = 0.0;

    return sym_A;
}
int main() {
    int N = 1000;

    cout << "Number of nodes: " << N << endl;
    
    int random_seed = 78375;



    auto distanceMatrix = create_symmetric_distance_matrix(N, random_seed);

    auto start = chrono::high_resolution_clock::now();
    auto mmjMatrix = ultra_fast_wide(distanceMatrix);
    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();

    cout << "Time used for calculating Min-Max-Jump (MMJ) distance matrix: " << time_used << " seconds\n";

    cout << "Print last 30 values of the first row of MMJ matrix: " << endl;
    const auto& row = mmjMatrix[0];
    for (size_t i = row.size() - 30; i < row.size(); ++i)
        cout << fixed << setprecision(1) << row[i] << " ";
    cout << "\n";    
    
    
    return 0;
}



Overwriting Reviewer_t5C9_code_cpp_version.cpp


In [3]:
!g++ -std=c++17 -O3 -march=native Reviewer_t5C9_code_cpp_version.cpp  -o tt -ltbb
!./tt

Number of nodes: 1000
Time used for calculating Min-Max-Jump (MMJ) distance matrix: 0.0110466 seconds
Print last 30 values of the first row of MMJ matrix: 
225.0 226.0 244.0 252.0 202.0 283.0 314.0 251.0 230.0 304.0 205.0 187.0 248.0 256.0 260.0 252.0 245.0 296.0 202.0 298.0 256.0 235.0 230.0 433.0 280.0 353.0 186.0 229.0 242.0 254.0 
